In [1]:
!pip install pyspark


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time

spark = SparkSession.builder \
    .appName("PySparkPractice") \
    .getOrCreate()

In [3]:
products_data = [
    ("P01", "Laptop", "Electronics", 60000),
    ("P02", "Monitor", "Electronics", 14000),
    ("P03", "Headphones", "Accessories", 3500),
    ("P04", "Printer", "Office", 17000),
    ("P05", "Router", "Networking", 4500),
    ("P06", "Tablet", "Electronics", 24000)
]

products_df = spark.createDataFrame(
    products_data,
    ["product_id", "product_name", "category", "list_price"]
)

stores_data = [
    ("S01", "Hyderabad", "South"),
    ("S02", "Bengaluru", "South"),
    ("S03", "Mumbai", "West"),
    ("S04", "Delhi", "North")
]

stores_df = spark.createDataFrame(
    stores_data,
    ["store_id", "city", "region"]
)

products_df.show()
stores_df.show()

+----------+------------+-----------+----------+
|product_id|product_name|   category|list_price|
+----------+------------+-----------+----------+
|       P01|      Laptop|Electronics|     60000|
|       P02|     Monitor|Electronics|     14000|
|       P03|  Headphones|Accessories|      3500|
|       P04|     Printer|     Office|     17000|
|       P05|      Router| Networking|      4500|
|       P06|      Tablet|Electronics|     24000|
+----------+------------+-----------+----------+

+--------+---------+------+
|store_id|     city|region|
+--------+---------+------+
|     S01|Hyderabad| South|
|     S02|Bengaluru| South|
|     S03|   Mumbai|  West|
|     S04|    Delhi| North|
+--------+---------+------+



In [4]:
sales_base_df = (
    spark.range(1, 100001)
    .withColumnRenamed("id", "order_id")
    .withColumn(
        "order_date",
        F.date_add(
            F.to_date(F.lit("2026-01-01")),
            (F.col("order_id") % 180).cast("int")
        )
    )
    .withColumn(
        "product_id",
        F.concat(
            F.lit("P"),
            F.lpad(
                ((F.col("order_id") % 6) + 1).cast("string"),
                2, "0"
            )
        )
    )
    .withColumn(
        "store_id",
        F.concat(
            F.lit("S"),
            F.lpad(
                ((F.floor(F.col("order_id") / 3) % 4) + 1)
                .cast("string"),
                2, "0"
            )
        )
    )
    .withColumn(
        "quantity",
        ((F.col("order_id") % 8) + 1).cast("int")
    )
    .withColumn(
        "discount_pct",
        F.when(
            F.col("order_id") % 19 == 0,
            F.lit(None).cast("int")
        ).otherwise(
            ((F.floor(F.col("order_id") / 5) % 4) * 5)
            .cast("int")
        )
    )
)

In [6]:
sales_df = (
    sales_base_df
    .withColumn(
        "order_status",
        F.when(F.col("order_id") % 11 == 0, "Cancelled")
        .when(F.col("order_id") % 7 == 0, "Returned")
        .otherwise("Completed")
    )
    .withColumn(
        "payment_method",
        F.when(F.col("order_id") % 3 == 0, "UPI")
        .when(F.col("order_id") % 3 == 1, "Card")
        .otherwise("Cash")
    )
    .withColumn(
        "salesperson_id",
        F.when(
            F.col("order_id") % 17 == 0,
            F.lit(None).cast("string")
        ).otherwise(
            F.concat(
                F.lit("EMP"),
                F.lpad(
                    ((F.col("order_id") % 12) + 1).cast("string"),
                    2, "0"
                )
            )
        )
    )
    .join(
        products_df.select(
            "product_id",
            F.col("list_price").alias("unit_price")
        ),
        "product_id", "left"
    )
    .select(
        "order_id", "order_date", "store_id",
        "product_id", "quantity", "unit_price",
        "discount_pct", "order_status",
        "payment_method", "salesperson_id"
    )
)

print("Total Records:", sales_df.count())
sales_df.printSchema()
sales_df.show(10)


Total Records: 100000
root
 |-- order_id: long (nullable = false)
 |-- order_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: long (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- order_status: string (nullable = false)
 |-- payment_method: string (nullable = false)
 |-- salesperson_id: string (nullable = true)

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|   Completed|           UPI|         EMP04|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5| 

**PART A**

In [7]:
sales_df.select("order_id", "product_id", "quantity", "unit_price").show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|       3|       P04|       4|     17000|
|       9|       P04|       2|     17000|
|      15|       P04|       8|     17000|
|      21|       P04|       6|     17000|
|       2|       P03|       3|      3500|
|       8|       P03|       1|      3500|
|      14|       P03|       7|      3500|
|      20|       P03|       5|      3500|
|       4|       P05|       5|      4500|
|      10|       P05|       3|      4500|
|      16|       P05|       1|      4500|
|       5|       P06|       6|     24000|
|      11|       P06|       4|     24000|
|      17|       P06|       2|     24000|
|       1|       P02|       2|     14000|
|       7|       P02|       8|     14000|
|      13|       P02|       6|     14000|
|      19|       P02|       4|     14000|
|       6|       P01|       7|     60000|
|      12|       P01|       5|     60000|
+--------+----------+--------+----

In [8]:
completed_high_qty_df = sales_df.filter(
    (F.col("order_status") == "Completed") & (F.col("quantity") > 5)
)

completed_high_qty_df.show()

completed_high_qty_df.count()

+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|order_status|payment_method|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------+--------------+--------------+
|      15|2026-01-16|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      39|2026-02-09|     S02|       P04|       8|     17000|          15|   Completed|           UPI|         EMP04|
|      45|2026-02-15|     S04|       P04|       6|     17000|           5|   Completed|           UPI|         EMP10|
|      69|2026-03-11|     S04|       P04|       6|     17000|           5|   Completed|           UPI|         EMP10|
|      38|2026-02-08|     S01|       P03|       7|      3500|        NULL|   Completed|          Cash|         EMP03|
|      62|2026-03-04|     S01|       P03|       7|      

29220

In [9]:
sales_gross_df = sales_df.withColumn(
    "gross_revenue",
    F.col("quantity") * F.col("unit_price")
)

sales_gross_df.select("order_id", "quantity", "unit_price", "gross_revenue").show()

+--------+--------+----------+-------------+
|order_id|quantity|unit_price|gross_revenue|
+--------+--------+----------+-------------+
|       3|       4|     17000|        68000|
|       9|       2|     17000|        34000|
|      15|       8|     17000|       136000|
|      21|       6|     17000|       102000|
|       2|       3|      3500|        10500|
|       8|       1|      3500|         3500|
|      14|       7|      3500|        24500|
|      20|       5|      3500|        17500|
|       4|       5|      4500|        22500|
|      10|       3|      4500|        13500|
|      16|       1|      4500|         4500|
|       5|       6|     24000|       144000|
|      11|       4|     24000|        96000|
|      17|       2|     24000|        48000|
|       1|       2|     14000|        28000|
|       7|       8|     14000|       112000|
|      13|       6|     14000|        84000|
|      19|       4|     14000|        56000|
|       6|       7|     60000|       420000|
|      12|

In [11]:
renamed_df = sales_df \
    .withColumnRenamed("order_status", "transaction_status") \
    .drop("payment_method")

renamed_df.printSchema()
renamed_df.show()

root
 |-- order_id: long (nullable = false)
 |-- order_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: long (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- transaction_status: string (nullable = false)
 |-- salesperson_id: string (nullable = true)

+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|transaction_status|salesperson_id|
+--------+----------+--------+----------+--------+----------+------------+------------------+--------------+
|       3|2026-01-04|     S02|       P04|       4|     17000|           0|         Completed|         EMP04|
|       9|2026-01-10|     S04|       P04|       2|     17000|           5|         Completed|         EMP10|
|      15|2026-01-16|     S02|       P04|       8|     17000|    

**PART B**

In [12]:
sales_df.groupBy("product_id") \
    .agg(F.sum("quantity").alias("total_quantity")) \
    .orderBy("product_id") \
    .show()

+----------+--------------+
|product_id|total_quantity|
+----------+--------------+
|       P01|         66668|
|       P02|         83336|
|       P03|         66667|
|       P04|         83334|
|       P05|         66665|
|       P06|         83330|
+----------+--------------+



In [13]:
sales_df.join(products_df, on="product_id", how="inner") \
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price")) \
    .groupBy("category") \
    .agg(F.sum("gross_revenue").alias("total_gross_revenue")) \
    .orderBy(F.col("total_gross_revenue").desc()) \
    .show()

+-----------+-------------------+
|   category|total_gross_revenue|
+-----------+-------------------+
|Electronics|         7166704000|
|     Office|         1416678000|
| Networking|          299992500|
|Accessories|          233334500|
+-----------+-------------------+



In [14]:
sales_df.join(stores_df, on="store_id", how="inner") \
    .groupBy("city") \
    .agg(F.count("*").alias("total_transactions")) \
    .orderBy(F.col("total_transactions").desc()) \
    .show()

+---------+------------------+
|     city|total_transactions|
+---------+------------------+
|Bengaluru|             25001|
|Hyderabad|             25001|
|   Mumbai|             24999|
|    Delhi|             24999|
+---------+------------------+



In [15]:
sales_df.join(stores_df, on="store_id", how="inner") \
    .groupBy("city") \
    .agg(F.count("*").alias("total_transactions")) \
    .orderBy(F.col("total_transactions").desc()) \
    .show()

+---------+------------------+
|     city|total_transactions|
+---------+------------------+
|Bengaluru|             25001|
|Hyderabad|             25001|
|   Mumbai|             24999|
|    Delhi|             24999|
+---------+------------------+



In [16]:
sales_df.join(products_df, on="product_id", how="inner") \
    .join(stores_df, on="store_id", how="inner") \
    .filter(F.col("order_status") == "Completed") \
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price")) \
    .groupBy("product_id", "product_name") \
    .agg(F.sum("gross_revenue").alias("total_gross_revenue")) \
    .orderBy(F.col("total_gross_revenue").desc()) \
    .show(5)

+----------+------------+-------------------+
|product_id|product_name|total_gross_revenue|
+----------+------------+-------------------+
|       P01|      Laptop|         3116940000|
|       P06|      Tablet|         1558416000|
|       P04|     Printer|         1103912000|
|       P02|     Monitor|          909076000|
|       P05|      Router|          233766000|
+----------+------------+-------------------+
only showing top 5 rows


**PART C**

In [17]:
sales_df.select(
    F.count(F.when(F.col("discount_pct").isNull(), 1)).alias("null_discount_pct"),
    F.count(F.when(F.col("salesperson_id").isNull(), 1)).alias("null_salesperson_id")
).show()

+-----------------+-------------------+
|null_discount_pct|null_salesperson_id|
+-----------------+-------------------+
|             5263|               5882|
+-----------------+-------------------+



In [18]:
sales_df.select(
    F.count(F.when(F.col("discount_pct").isNull(), 1)).alias("null_discount_pct"),
    F.count(F.when(F.col("salesperson_id").isNull(), 1)).alias("null_salesperson_id")
).show()

+-----------------+-------------------+
|null_discount_pct|null_salesperson_id|
+-----------------+-------------------+
|             5263|               5882|
+-----------------+-------------------+



In [19]:
sales_with_rep_df = sales_df.na.drop(subset=["salesperson_id"])

sales_with_rep_df.count()

94118

In [21]:
order_size_df = sales_df.withColumn(
    "order_size",
    F.when(F.col("quantity").between(1, 3), "Small")
     .when(F.col("quantity").between(4, 6), "Medium")
     .otherwise("Large")
)

order_size_df.groupBy("order_size").count().orderBy("order_size").show()

+----------+-----+
|order_size|count|
+----------+-----+
|     Large|25000|
|    Medium|37500|
|     Small|37500|
+----------+-----+



**PART D**

In [22]:
category_product_df = sales_df.join(products_df, on="product_id", how="inner") \
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price")) \
    .groupBy("category", "product_name") \
    .agg(F.sum("gross_revenue").alias("total_gross_revenue"))

category_window = Window.partitionBy("category") \
    .orderBy(F.col("total_gross_revenue").desc())

category_product_df.withColumn("revenue_rank", F.dense_rank().over(category_window)) \
    .orderBy("category", "revenue_rank") \
    .show()

+-----------+------------+-------------------+------------+
|   category|product_name|total_gross_revenue|revenue_rank|
+-----------+------------+-------------------+------------+
|Accessories|  Headphones|          233334500|           1|
|Electronics|      Laptop|         4000080000|           1|
|Electronics|      Tablet|         1999920000|           2|
|Electronics|     Monitor|         1166704000|           3|
| Networking|      Router|          299992500|           1|
|     Office|     Printer|         1416678000|           1|
+-----------+------------+-------------------+------------+



In [24]:
running_window = Window.partitionBy("store_id") \
    .orderBy("order_date", "order_id") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

sales_df.withColumn("gross_revenue", F.col("quantity") * F.col("unit_price")) \
    .withColumn("running_gross_revenue", F.sum("gross_revenue").over(running_window)) \
    .select("store_id", "order_date", "order_id", "gross_revenue", "running_gross_revenue") \
    .orderBy("store_id", "order_date", "order_id") \
    .show()

+--------+----------+--------+-------------+---------------------+
|store_id|order_date|order_id|gross_revenue|running_gross_revenue|
+--------+----------+--------+-------------+---------------------+
|     S01|2026-01-01|     180|       300000|               300000|
|     S01|2026-01-01|     360|        60000|               360000|
|     S01|2026-01-01|     540|       300000|               660000|
|     S01|2026-01-01|     720|        60000|               720000|
|     S01|2026-01-01|     900|       300000|              1020000|
|     S01|2026-01-01|    1080|        60000|              1080000|
|     S01|2026-01-01|    1260|       300000|              1380000|
|     S01|2026-01-01|    1440|        60000|              1440000|
|     S01|2026-01-01|    1620|       300000|              1740000|
|     S01|2026-01-01|    1800|        60000|              1800000|
|     S01|2026-01-01|    1980|       300000|              2100000|
|     S01|2026-01-01|    2160|        60000|              2160

In [25]:
sales_df.filter(F.col("order_status") == "Completed") \
    .withColumn("month", F.month("order_date")) \
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price")) \
    .groupBy("month") \
    .agg(F.sum("gross_revenue").alias("monthly_revenue")) \
    .orderBy("month") \
    .show()

+-----+---------------+
|month|monthly_revenue|
+-----+---------------+
|    1|     1278728500|
|    2|     1027585500|
|    3|     1212087500|
|    4|     1218921000|
|    5|     1277560500|
|    6|     1089027500|
+-----+---------------+



**PART E**

In [27]:
step1 = sales_df.filter(F.col("order_status") == "Completed")
print("filter defined")

step2 = step1.withColumn("gross_revenue", F.col("quantity") * F.col("unit_price"))
print("withColumn defined")

step3 = step2.select("order_id", "product_id", "quantity", "unit_price", "gross_revenue")
print("select defined")

step3.show()

filter defined
withColumn defined
select defined
+--------+----------+--------+----------+-------------+
|order_id|product_id|quantity|unit_price|gross_revenue|
+--------+----------+--------+----------+-------------+
|       3|       P04|       4|     17000|        68000|
|       9|       P04|       2|     17000|        34000|
|      15|       P04|       8|     17000|       136000|
|       2|       P03|       3|      3500|        10500|
|       8|       P03|       1|      3500|         3500|
|      20|       P03|       5|      3500|        17500|
|      26|       P03|       3|      3500|        10500|
|       4|       P05|       5|      4500|        22500|
|      10|       P05|       3|      4500|        13500|
|      16|       P05|       1|      4500|         4500|
|       5|       P06|       6|     24000|       144000|
|      17|       P06|       2|     24000|        48000|
|      23|       P06|       8|     24000|       192000|
|       1|       P02|       2|     14000|        28000|

In [28]:
product_totals_df = sales_df.groupBy("product_id") \
    .agg(F.sum(F.col("quantity") * F.col("unit_price")).alias("total_gross_revenue"))

product_totals_df.explain()

results = product_totals_df.orderBy("product_id").collect()

for row in results:
    print(row["product_id"], row["total_gross_revenue"])

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[product_id#33], functions=[sum((cast(quantity#35 as bigint) * unit_price#89L))])
   +- HashAggregate(keys=[product_id#33], functions=[partial_sum((cast(quantity#35 as bigint) * unit_price#89L))])
      +- Project [product_id#33, quantity#35, unit_price#89L]
         +- SortMergeJoin [product_id#33], [product_id#0], LeftOuter
            :- Sort [product_id#33 ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=6502]
            :     +- Project [concat(P, lpad(cast(((id#30L % 6) + 1) as string), 2, 0)) AS product_id#33, cast(((id#30L % 8) + 1) as int) AS quantity#35]
            :        +- Range (1, 100001, step=1, splits=2)
            +- Sort [product_id#0 ASC NULLS FIRST], false, 0
               +- Exchange hashpartitioning(product_id#0, 200), ENSURE_REQUIREMENTS, [plan_id=6503]
                  +- Project [product_id#0, list_price#3

**PART F**

In [29]:
completed_df = sales_df.filter(F.col("order_status") == "Completed")

completed_df.cache()

start = time.time()
first_count = completed_df.count()
first_time = time.time() - start

start = time.time()
second_count = completed_df.count()
second_time = time.time() - start

print("First count:", first_count, "-", round(first_time, 4), "seconds")
print("Second count:", second_count, "-", round(second_time, 4), "seconds")

First count: 77923 - 5.9916 seconds
Second count: 77923 - 1.4641 seconds


In [30]:
high_qty_df = sales_df.filter(F.col("quantity") > 5)

high_qty_df.persist(StorageLevel.MEMORY_AND_DISK)

print("Count:", high_qty_df.count())

high_qty_df.groupBy("product_id").count().orderBy("product_id").show()

print(high_qty_df.storageLevel)

Count: 37500
+----------+-----+
|product_id|count|
+----------+-----+
|       P01| 4167|
|       P02| 8334|
|       P03| 4167|
|       P04| 8333|
|       P05| 4166|
|       P06| 8333|
+----------+-----+

Disk Memory Serialized 1x Replicated


In [31]:
high_qty_df.unpersist(blocking=True)

print("Is cached:", high_qty_df.is_cached)

Is cached: False


In [32]:
category_agg_df = sales_df.join(products_df, on="product_id", how="inner") \
    .groupBy("category") \
    .agg(F.sum(F.col("quantity") * F.col("unit_price")).alias("total_gross_revenue"))

category_agg_df.cache()

category_agg_df.count()

category_agg_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [category#1799, total_gross_revenue#1801L]
      +- InMemoryRelation [category#1799, total_gross_revenue#1801L], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- AdaptiveSparkPlan isFinalPlan=true
               +- == Final Plan ==
                  ResultQueryStage 5
                  +- *(7) HashAggregate(keys=[category#1799], functions=[sum((cast(quantity#35 as bigint) * unit_price#89L))])
                     +- ShuffleQueryStage 4
                        +- Exchange hashpartitioning(category#1799, 200), ENSURE_REQUIREMENTS, [plan_id=7528]
                           +- *(6) HashAggregate(keys=[category#1799], functions=[partial_sum((cast(quantity#35 as bigint) * unit_price#89L))])
                              +- *(6) Project [quantity#35, unit_price#89L, category#1799]
                                 +- *(6) SortMergeJoin [product_id#33], [product_id#1797], Inner
                      

**PART G**

In [33]:
print("Existing partitions:", sales_df.rdd.getNumPartitions())

sales_8_df = sales_df.repartition(8)

print("New partitions:", sales_8_df.rdd.getNumPartitions())

Existing partitions: 2
New partitions: 8


In [34]:
sales_8_df.withColumn("partition_id", F.spark_partition_id()) \
    .groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|12500|
|           1|12500|
|           2|12500|
|           3|12500|
|           4|12500|
|           5|12500|
|           6|12500|
|           7|12500|
+------------+-----+



In [35]:
sales_store_df = sales_df.repartition(6, "store_id")

print("Partitions:", sales_store_df.rdd.getNumPartitions())

sales_store_df.withColumn("partition_id", F.spark_partition_id()) \
    .groupBy("partition_id", "store_id") \
    .count() \
    .orderBy("partition_id", "store_id") \
    .show()

Partitions: 6
+------------+--------+-----+
|partition_id|store_id|count|
+------------+--------+-----+
|           1|     S03|24999|
|           3|     S02|25001|
|           4|     S01|25001|
|           5|     S04|24999|
+------------+--------+-----+



In [37]:
sales_12_df = sales_df.repartition(12)

coalesced_4_df = sales_12_df.coalesce(4)
repartitioned_4_df = sales_12_df.repartition(4)

print( sales_12_df.rdd.getNumPartitions())
print(coalesced_4_df.rdd.getNumPartitions())
print(repartitioned_4_df.rdd.getNumPartitions())

12
4
4


In [38]:
print("COALESCE PLAN")
coalesced_4_df.explain()

print("REPARTITION PLAN")
repartitioned_4_df.explain()

COALESCE PLAN
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 4
   +- Coalesce 4
      +- ShuffleQueryStage 3
         +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=9454]
            +- *(3) Project [order_id#31L, order_date#32, store_id#34, product_id#33, quantity#35, unit_price#89L, discount_pct#36, order_status#86, payment_method#87, salesperson_id#88]
               +- *(3) BroadcastHashJoin [product_id#33], [product_id#0], LeftOuter, BuildRight, false
                  :- AQEShuffleRead local
                  :  +- ShuffleQueryStage 0
                  :     +- Exchange hashpartitioning(product_id#33, 200), ENSURE_REQUIREMENTS, [plan_id=9354]
                  :        +- *(1) Project [id#30L AS order_id#31L, date_add(2026-01-01, cast((id#30L % 180) as int)) AS order_date#32, concat(P, lpad(cast(((id#30L % 6) + 1) as string), 2, 0)) AS product_id#33, concat(S, lpad(cast(((FLOOR((cast(id#30L as double) / 3.0)) %

In [39]:
sales_df.repartition(4) \
    .write.mode("overwrite") \
    .option("header", True) \
    .csv("sales_csv_output")

In [40]:
import os

print(sorted(f for f in os.listdir("sales_csv_output") if f.startswith("part-")))

spark.read.csv("sales_csv_output", header=True).count()

['part-00000-fe50b468-52ab-44b9-b837-89e28e564439-c000.csv', 'part-00001-fe50b468-52ab-44b9-b837-89e28e564439-c000.csv', 'part-00002-fe50b468-52ab-44b9-b837-89e28e564439-c000.csv', 'part-00003-fe50b468-52ab-44b9-b837-89e28e564439-c000.csv']


100000

**Final** **Challenge**

In [41]:
import os

print(sorted(f for f in os.listdir("sales_csv_output") if f.startswith("part-")))

spark.read.csv("sales_csv_output", header=True).count()

['part-00000-fe50b468-52ab-44b9-b837-89e28e564439-c000.csv', 'part-00001-fe50b468-52ab-44b9-b837-89e28e564439-c000.csv', 'part-00002-fe50b468-52ab-44b9-b837-89e28e564439-c000.csv', 'part-00003-fe50b468-52ab-44b9-b837-89e28e564439-c000.csv']


100000

In [44]:
pipeline_df = sales_df \
    .filter(F.col("order_status") == "Completed") \
    .na.fill({"discount_pct": 0}) \
    .join(products_df, on="product_id", how="inner") \
    .join(stores_df, on="store_id", how="inner") \
    .withColumn("gross_revenue", F.col("quantity") * F.col("unit_price")) \
    .withColumn(
        "discounted_revenue",
        F.col("gross_revenue") * (1 - F.col("discount_pct") / 100)
    )

region_category_df = pipeline_df.groupBy("region", "category").agg(
    F.sum("gross_revenue").alias("total_gross_revenue"),
    F.round(F.sum("discounted_revenue"), 2).alias("total_discounted_revenue"),
    F.count("*").alias("transaction_count")
)

rank_window = Window.partitionBy("region") \
    .orderBy(F.col("total_discounted_revenue").desc())

final_df = region_category_df.withColumn(
    "category_rank",
    F.dense_rank().over(rank_window)
)

In [45]:
final_df.cache()

final_df.count()

final_df.orderBy("region", "category_rank").show()

+------+-----------+-------------------+------------------------+-----------------+-------------+
|region|   category|total_gross_revenue|total_discounted_revenue|transaction_count|category_rank|
+------+-----------+-------------------+------------------------+-----------------+-------------+
| North|Electronics|          934848000|              8.551824E8|             6492|            1|
| North|     Office|          441388000|               4.12114E8|             6492|            2|
| North| Networking|          146097000|            1.35026775E8|             6494|            3|
| South|Electronics|         2156236000|             2.0242794E9|            19484|            1|
| South|     Office|          662524000|              6.060058E8|             6495|            2|
| South|Accessories|          113631000|            1.05018025E8|             6494|            3|
| South| Networking|           87669000|             8.2684125E7|             6494|            4|
|  West|Electronics|

In [46]:
final_df.repartition(2) \
    .write.mode("overwrite") \
    .parquet("final_region_category_revenue")

In [47]:
print(sorted(f for f in os.listdir("final_region_category_revenue") if f.startswith("part-")))

spark.read.parquet("final_region_category_revenue").orderBy("region", "category_rank").show()

['part-00000-25b55a56-a32e-45c5-b28d-07daf7d7dc0f-c000.snappy.parquet', 'part-00001-25b55a56-a32e-45c5-b28d-07daf7d7dc0f-c000.snappy.parquet']
+------+-----------+-------------------+------------------------+-----------------+-------------+
|region|   category|total_gross_revenue|total_discounted_revenue|transaction_count|category_rank|
+------+-----------+-------------------+------------------------+-----------------+-------------+
| North|Electronics|          934848000|              8.551824E8|             6492|            1|
| North|     Office|          441388000|               4.12114E8|             6492|            2|
| North| Networking|          146097000|            1.35026775E8|             6494|            3|
| South|Electronics|         2156236000|             2.0242794E9|            19484|            1|
| South|     Office|          662524000|              6.060058E8|             6495|            2|
| South|Accessories|          113631000|            1.05018025E8|        

In [48]:
completed_df.unpersist(blocking=True)
category_agg_df.unpersist(blocking=True)
final_df.unpersist(blocking=True)


DataFrame[region: string, category: string, total_gross_revenue: bigint, total_discounted_revenue: double, transaction_count: bigint, category_rank: int]